# Molecular dynamics of one ethanol molecule
Use ASE to integrate motion and MACE-OFF23-small to evaluate energies and forces.
Learn how a thermostat differs from energy-conserving dynamics, and inspect a molecular trajectory.

Run from this notebook's directory with a Python environment containing MACE >=0.3.14,
ASE, PyTorch and the demo visualization dependencies. The model is in `../models`.
We use flexible bonds, a 0.25 fs timestep, and CPU execution. These are short teaching runs,
not converged conformational populations. All MD steps are visible below.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.optimize import BFGS
from mace.calculators import MACECalculator

sys.path.insert(0, str(Path.cwd().parent.parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '1')))
MODEL = Path('../../models/MACE-OFF23_small.model')
calc = MACECalculator(model_paths=str(MODEL), device='cpu', default_dtype='float64')


## 1. Prepare ethanol
Predict which motions will be fastest: bond vibrations or torsional rotations?

In [ ]:
from ase.build import molecule
atoms = molecule('CH3CH2OH')
atoms.info.update(total_charge=0, total_spin=0)
atoms.calc = calc
opt = BFGS(atoms, logfile=str(OUTPUT / 'relax.log'))
assert opt.run(fmax=0.02, steps=200), 'Inspect relax.log before running MD'
view(atoms)


## 2. Initialize velocities and time a short NVT run
The same random seed makes this example reproducible. Temperature fluctuates strongly for only nine atoms. We remove initial net translation; the Langevin thermostat acts on all Cartesian degrees of freedom, so center-of-mass motion can subsequently develop.

In [ ]:
T = 300
DT_FS = 0.25
STEPS = 2000
rng = np.random.default_rng(7)
MaxwellBoltzmannDistribution(atoms, temperature_K=T, rng=rng)
Stationary(atoms)
dyn = Langevin(atoms, DT_FS * units.fs, temperature_K=T,
               friction=0.01 / units.fs, fixcm=False, rng=rng,
               trajectory=str(OUTPUT / 'nvt.traj'), loginterval=10)
start = time.perf_counter()
dyn.run(10)
seconds_per_step = (time.perf_counter() - start) / 10
print(f'Estimated {STEPS}-step runtime: {seconds_per_step * STEPS / 60:.1f} min')


## 3. Run NVT
The thermostat exchanges energy with the molecule. Total energy need not be constant in this stage. Adjust STEPS using the timing above.

In [ ]:
dyn.run(STEPS)
frames = read(OUTPUT / 'nvt.traj', index=':')
t = np.arange(len(frames)) * 10 * DT_FS / 1000
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t, [a.get_temperature() for a in frames])
ax[0].axhline(T, color='k', linestyle='--')
ax[0].set(xlabel='Time (ps)', ylabel='Temperature (K)')
ax[1].plot(t, [a.get_potential_energy() for a in frames])
ax[1].set(xlabel='Time (ps)', ylabel='Potential energy (eV)')
plt.show()
view(frames[::5], animate=True)


## 4. Remove the thermostat: NVE
Continue with the final NVT positions and velocities. Compare energy drift with the same initial state using half the timestep and twice the steps.

In [ ]:
from ase.md.verlet import VelocityVerlet
write(OUTPUT / 'nve_start.traj', atoms)
nve = VelocityVerlet(atoms, DT_FS * units.fs,
                     trajectory=str(OUTPUT / 'nve.traj'), loginterval=10)
nve.run(1000)
nve_frames = read(OUTPUT / 'nve.traj', index=':')
energy = np.array([a.get_total_energy() for a in nve_frames])
plt.plot(np.arange(len(energy)) * 10 * DT_FS / 1000, 1000 * (energy - energy[0]))
plt.xlabel('Time (ps)'); plt.ylabel('Total energy change (meV)'); plt.show()


## Discuss
1. Why does instantaneous temperature fluctuate so much?
2. Why is total energy conserved approximately in NVE but not in NVT?
3. Does a short animation demonstrate equilibrium conformer populations?

References: [ASE MD](https://ase-lib.org/ase/md.html),
[MACE foundation models](https://github.com/ACEsuit/mace-foundations).